# 💵 Ejercicio 1: Predicción del Precio del Dólar
## Minería de Datos — CRISP-DM | Regresión Lineal Múltiple

**Variable dependiente:** `Precio_Dolar`  
**Variables independientes:** `Dia`, `Inflacion`, `Tasa_interes`


## 1. Comprensión del Negocio
El objetivo es predecir el precio del dólar colombiano (COP) en función del número de día, la tasa de inflación diaria y la tasa de interés. Este tipo de modelo puede ser útil para analistas financieros y economistas.

## 2. Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

os.makedirs("modelos", exist_ok=True)
os.makedirs("graficas", exist_ok=True)
print("✅ Librerías importadas correctamente")

## 3. Comprensión y Exploración de los Datos

In [ ]:
df = pd.read_csv("dolar_data.csv")
print(f"Dimensiones del dataset: {df.shape}")
df.head(10)

In [ ]:
print("📋 Estadísticas descriptivas:")
df.describe().round(4)

In [ ]:
print("🔍 Valores nulos:")
print(df.isnull().sum())
print(f"\nTipos de datos:")
print(df.dtypes)

In [ ]:
print("📊 Matriz de correlación:")
corr = df.corr()
print(corr.round(4))

## 4. Preparación de los Datos

In [ ]:
# Separar variables independientes y dependiente
X = df[['Dia', 'Inflacion', 'Tasa_interes']].values
y = df['Precio_Dolar'].values

# División entrenamiento / prueba (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Tamaño entrenamiento: {X_train.shape[0]} registros")
print(f"Tamaño prueba       : {X_test.shape[0]} registros")

## 5. Modelado — Regresión Lineal Múltiple

In [ ]:
# Entrenar el modelo
model = LinearRegression()
model.fit(X_train, y_train)

# Mostrar coeficientes
print("📊 Coeficientes del modelo:")
print(f"  Intercepto   : {model.intercept_:.4f}")
for feat, coef in zip(['Dia', 'Inflacion', 'Tasa_interes'], model.coef_):
    print(f"  {feat:15s}: {coef:+.4f}")

print(f"\n  Ecuación: Precio_Dolar = {model.intercept_:.2f} + {model.coef_[0]:.4f}·Dia + {model.coef_[1]:.4f}·Inflacion + {model.coef_[2]:.4f}·Tasa_interes")

### Interpretación de coeficientes
- **Dia**: Por cada día adicional, el precio del dólar cambia en el valor del coeficiente (COP). Refleja la tendencia temporal del tipo de cambio.
- **Inflacion**: Una unidad más de inflación afecta el precio. Generalmente una mayor inflación debilita la moneda local.
- **Tasa_interes**: Tasas más altas pueden atraer capital extranjero, afectando el tipo de cambio.

## 6. Evaluación del Modelo

In [ ]:
# Predicciones sobre el conjunto de prueba
y_pred = model.predict(X_test)

# Métricas de desempeño
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print("📈 Desempeño del modelo (conjunto de prueba):")
print(f"  MSE  : {mse:.4f}")
print(f"  RMSE : {rmse:.4f} COP")
print(f"  R²   : {r2:.4f}  →  explica el {r2*100:.2f}% de la varianza")

In [ ]:
# Importancia relativa de variables (coeficientes estandarizados)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[['Dia', 'Inflacion', 'Tasa_interes']])
model_std = LinearRegression().fit(X_scaled, y)

print("🔬 Importancia de variables (coeficientes estandarizados):")
for feat, coef in sorted(zip(['Dia','Inflacion','Tasa_interes'], np.abs(model_std.coef_)), key=lambda x: -x[1]):
    print(f"  {feat:15s}: {coef:.4f}")

## 7. Visualizaciones

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
features = ['Dia', 'Inflacion', 'Tasa_interes']
colores  = ['#2196F3', '#4CAF50', '#FF5722']

for ax, feat, color in zip(axes, features, colores):
    ax.scatter(df[feat], df['Precio_Dolar'], alpha=0.4, s=10, color=color)
    m, b = np.polyfit(df[feat], df['Precio_Dolar'], 1)
    x_line = np.linspace(df[feat].min(), df[feat].max(), 100)
    ax.plot(x_line, m*x_line + b, color='red', linewidth=1.5, label='Tendencia')
    ax.set_xlabel(feat, fontsize=11)
    ax.set_ylabel('Precio_Dolar', fontsize=11)
    ax.set_title(f'{feat} vs Precio_Dolar', fontsize=12)
    ax.legend(); ax.grid(True, alpha=0.3)

fig.suptitle('Ejercicio 1: Variables vs Precio del Dólar', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('graficas/dolar_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfica guardada en graficas/dolar_scatter.png")

In [ ]:
# Valores reales vs predichos
plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_pred, alpha=0.5, color='#1565C0', s=15)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=1.5, label='Predicción perfecta')
plt.xlabel('Valor Real (COP)', fontsize=12)
plt.ylabel('Valor Predicho (COP)', fontsize=12)
plt.title(f'Real vs Predicho — R² = {r2:.4f}', fontsize=13)
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Exportación del Modelo

In [ ]:
# Guardar el modelo entrenado con joblib
joblib.dump(model, 'modelos/modelo_dolar.joblib')
print("✅ Modelo guardado en: modelos/modelo_dolar.joblib")

# Verificar carga del modelo
modelo_cargado = joblib.load('modelos/modelo_dolar.joblib')
prueba = modelo_cargado.predict([[250, 0.020, 5.0]])
print(f"\n🔮 Prueba de predicción (Día=250, Inflación=0.020, Tasa=5.0%): ${prueba[0]:,.2f} COP")

## 9. Conclusiones
- El modelo de Regresión Lineal Múltiple logra un **R² = 0.9963**, explicando el 99.63% de la varianza del precio del dólar.
- La variable **Dia** es el predictor más importante, capturando la tendencia temporal ascendente del tipo de cambio.
- El RMSE de ~48.75 COP es muy bajo en relación al rango de precios, confirmando la alta precisión del modelo.
- El modelo cumple el criterio docente de R² > 0.90 ✅